In [2]:
import pandas as pd

df = pd.read_csv("../PLN/data/noticias_games (2).csv")
df.head()

,url,titulo,descricao,autor,tags,texto,data
0,https://www.eurogamer.pt/tecmo-bowl-sera-adapt...,Tecmo Bowl será adaptado para filme,"Tecmo Bowl, clássico da Koei Tecmo lançado no ...",NaN,"Android, iOS","Tecmo Bowl, clássico da Koei Tecmo lançado no ...",2026-02-03T16:35:31.309576+00:00
1,https://www.eurogamer.pt/fas-adoram-super-mari...,Fãs adoram Super Mario Galaxy: O Filme,Super Mario Galaxy: O Filme chegou finalmente ...,NaN,"Action Adventure, Platformer, Nintendo Switch,...",Super Mario Galaxy: O Filme chegou finalmente ...,2026-04-02T07:40:11.552201+00:00
2,https://www.eurogamer.pt/pokemon-firered-leafg...,Pokémon FireRed & LeafGreen inclui conteúdo au...,Pokémon FireRed & LeafGreen para a Nintendo Sw...,NaN,"Action Adventure, Pokémon FireRed and LeafGree...",Pokémon FireRed & LeafGreen para a Nintendo Sw...,2026-03-01T09:15:15.110566+00:00
3,https://www.eurogamer.pt/switch-2-ja-alcancou-...,Switch 2 já alcançou 58% das vendas da PS5 em ...,"O recente relatório do governo japonês, sobre ...",NaN,"Third person, Racing, Nintendo, Nintendo Switc...","O recente relatório do governo japonês, sobre ...",2026-02-09T13:33:14.368225+00:00
4,https://www.eurogamer.pt/produtora-de-slay-the...,Produtora de Slay the Spire 2 detesta micro-tr...,"Slay the Spire 2 chegou com grande sucesso, em...",NaN,"Multiplayer Cooperative, Indie, PC, Slay the S...","Slay the Spire 2 chegou com grande sucesso, em...",2026-03-10T14:42:15.059919+00:00


In [3]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto"].apply(limpar_texto)

df[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,"Tecmo Bowl, clássico da Koei Tecmo lançado no ...",tecmo bowl classico da koei tecmo lancado no f...
1,Super Mario Galaxy: O Filme chegou finalmente ...,super mario galaxy o filme chegou finalmente a...
2,Pokémon FireRed & LeafGreen para a Nintendo Sw...,pokemon firered leafgreen para a nintendo swit...
3,"O recente relatório do governo japonês, sobre ...",o recente relatorio do governo japones sobre o...
4,"Slay the Spire 2 chegou com grande sucesso, em...",slay the spire 2 chegou com grande sucesso em ...


In [4]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,tecmo bowl classico da koei tecmo lancado no f...,"[tecmo, bowl, classico, koei, tecmo, lancado, ...",tecmo bowl classico koei tecmo lancado final a...
1,super mario galaxy o filme chegou finalmente a...,"[super, mario, galaxy, filme, chegou, finalmen...",super mario galaxy filme chegou finalmente cin...
2,pokemon firered leafgreen para a nintendo swit...,"[pokemon, firered, leafgreen, nintendo, switch...",pokemon firered leafgreen nintendo switch incl...
3,o recente relatorio do governo japones sobre o...,"[recente, relatorio, governo, japones, sobre, ...",recente relatorio governo japones sobre gastos...
4,slay the spire 2 chegou com grande sucesso em ...,"[slay, the, spire, 2, chegou, grande, sucesso,...",slay the spire 2 chegou grande sucesso acesso ...


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,000mo,003,005,007,00h,01,011,012,...,zstd,zu,zuo,zur,zuwoyou,zwei,zwigqfdrl8,zygarde,zynga,zysppc5w4p
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:

colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]
df_bow = df_bow.drop(columns=colunas_com_numeros)

freq = df_bow.sum(axis=0)

colunas_validas = freq[freq > 2].index

df_bow = df_bow[colunas_validas]

print(f"{len(colunas_com_numeros)} colunas com números removidas")
print(f"{len(freq) - len(colunas_validas)} colunas raras removidas")

df_bow.head()

1232 colunas com números removidas
12226 colunas raras removidas


,aaa,aang,aaron,abaixo,abandona,abandonado,abandonar,abandono,abandonou,abby,...,zona,zonas,zone,zootopia,zootropolis,zor,zoro,zur,zwei,zygarde
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
metadados = df[["data", "tags", "titulo", "url"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,tags,titulo,url,bow_aaa,bow_aang,bow_aaron,bow_abaixo,bow_abandona,bow_abandonado,...,bow_zona,bow_zonas,bow_zone,bow_zootopia,bow_zootropolis,bow_zor,bow_zoro,bow_zur,bow_zwei,bow_zygarde
0,2026-02-03T16:35:31.309576+00:00,"Android, iOS",Tecmo Bowl será adaptado para filme,https://www.eurogamer.pt/tecmo-bowl-sera-adapt...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2026-04-02T07:40:11.552201+00:00,"Action Adventure, Platformer, Nintendo Switch,...",Fãs adoram Super Mario Galaxy: O Filme,https://www.eurogamer.pt/fas-adoram-super-mari...,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2026-03-01T09:15:15.110566+00:00,"Action Adventure, Pokémon FireRed and LeafGree...",Pokémon FireRed & LeafGreen inclui conteúdo au...,https://www.eurogamer.pt/pokemon-firered-leafg...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2026-02-09T13:33:14.368225+00:00,"Third person, Racing, Nintendo, Nintendo Switc...",Switch 2 já alcançou 58% das vendas da PS5 em ...,https://www.eurogamer.pt/switch-2-ja-alcancou-...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2026-03-10T14:42:15.059919+00:00,"Multiplayer Cooperative, Indie, PC, Slay the S...",Produtora de Slay the Spire 2 detesta micro-tr...,https://www.eurogamer.pt/produtora-de-slay-the...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 9180


bow_jogo         2049
bow_the          1129
bow_jogadores    1022
bow_jogos        1020
bow_ver           862
bow_serie         815
bow_nintendo      803
bow_novo          747
bow_vai           722
bow_of            721
dtype: int64

In [9]:
frequencia_palavras.tail(10)

bow_errado         3
bow_eric           3
bow_envolveu       3
bow_recusou        3
bow_redesenhada    3
bow_redfield       3
bow_equacao        3
bow_epicos         3
bow_epicas         3
bow_zygarde        3
dtype: int64

In [10]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

jogo         738
ver          610
ainda        500
jogos        498
novo         493
jogadores    483
vai          471
the          455
youtube      411
nova         408
dtype: int64

In [11]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
833,Nacon revela novidades em março,29
1132,Nintendo Direct: Partner Showcase - vê em direto,30
11,A Marvel começa a lembrar-se que os X-Men existem,33
984,Nintendo Treehouse: Live anunciada,36
653,Primeiras imagens de Pokémon Winds & Waves,37
970,Lords of the Fallen vendeu 2.5 milhões de cópias,37
748,Pokémon XD: Gale of Darkness chegará à Switch 2,39
7,Pokémon Pokopia celebra o Dia das Mentiras,40
887,Nintendo Indie World anunciada,40
645,Jogo das cuecas está a caminho da PS5 e PS4,41


In [12]:
matriz_bow_df = df_final[colunas_bow].copy()
matriz_bow_df.columns = matriz_bow_df.columns.str.replace("bow_", "", regex=False)

matriz_bow_df.head()

,aaa,aang,aaron,abaixo,abandona,abandonado,abandonar,abandono,abandonou,abby,...,zona,zonas,zone,zootopia,zootropolis,zor,zoro,zur,zwei,zygarde
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
total_palavras_por_noticia = matriz_bow_df.sum(axis=1)

matriz_tf = matriz_bow_df.div(total_palavras_por_noticia, axis=0)

matriz_tf.head()

,aaa,aang,aaron,abaixo,abandona,abandonado,abandonar,abandono,abandonou,abby,...,zona,zonas,zone,zootopia,zootropolis,zor,zoro,zur,zwei,zygarde
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.016529,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
noticia = 0

print(df_final.loc[noticia, "titulo"])

matriz_tf.loc[noticia].sort_values(ascending=False).head(10)

Tecmo Bowl será adaptado para filme


tecmo        0.065789
bowl         0.052632
live         0.039474
action       0.039474
jogo         0.026316
produzido    0.026316
filme        0.026316
liderada     0.026316
classico     0.026316
criadores    0.013158
Name: 0, dtype: float64

In [24]:
import numpy as np
N = len(matriz_bow_df)
documentos_por_termo = (matriz_bow_df > 0).sum(axis=0)

idf = np.log(N / documentos_por_termo)

df_idf = pd.DataFrame({
    "documentos_com_o_termo": documentos_por_termo,
    "idf": idf
}).sort_values("idf", ascending=False)

df_idf.head(10)

,documentos_com_o_termo,idf
druenen,1,7.130099
jaeger,1,7.130099
trigger,1,7.130099
trials,1,7.130099
dell,1,7.130099
jam,1,7.130099
jantes,1,7.130099
millennium,1,7.130099
fryvaldsky,1,7.130099
japanawaits,1,7.130099


In [25]:
df_idf.tail(10)

,documentos_com_o_termo,idf
nova,408,1.118831
youtube,411,1.111505
the,455,1.009801
vai,471,0.975240
jogadores,483,0.950082
novo,493,0.929589
jogos,498,0.919498
ainda,500,0.915490
ver,610,0.716640
jogo,738,0.526155


In [26]:
matriz_tfidf = matriz_tf.mul(idf, axis=1)

matriz_tfidf.head()

,aaa,aang,aaron,abaixo,abandona,abandonado,abandonar,abandono,abandonou,abby,...,zona,zonas,zone,zootopia,zootropolis,zor,zoro,zur,zwei,zygarde
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.057727,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
noticia = 10

print(df_final.loc[noticia, "titulo"])

matriz_tfidf.loc[noticia].sort_values(ascending=False).head(10)

O novo grande exclusivo PS5 está quase a chegar


saros              0.201965
housemarque        0.120014
frenetica          0.112700
funcionalidades    0.110623
uso                0.097923
dualsense          0.097631
pronta             0.085914
fabricas           0.062433
reproducao         0.062433
returnal           0.062433
Name: 10, dtype: float64

In [28]:
tfidf_com_prefixo = matriz_tfidf.add_prefix("tfidf_").reset_index(drop=True)

df_final_tfidf = pd.concat([metadados, tfidf_com_prefixo], axis=1)

df_final_tfidf.head()

,data,tags,titulo,url,tfidf_aaa,tfidf_aang,tfidf_aaron,tfidf_abaixo,tfidf_abandona,tfidf_abandonado,...,tfidf_zona,tfidf_zonas,tfidf_zone,tfidf_zootopia,tfidf_zootropolis,tfidf_zor,tfidf_zoro,tfidf_zur,tfidf_zwei,tfidf_zygarde
0,2026-02-03T16:35:31.309576+00:00,"Android, iOS",Tecmo Bowl será adaptado para filme,https://www.eurogamer.pt/tecmo-bowl-sera-adapt...,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-04-02T07:40:11.552201+00:00,"Action Adventure, Platformer, Nintendo Switch,...",Fãs adoram Super Mario Galaxy: O Filme,https://www.eurogamer.pt/fas-adoram-super-mari...,0.0,0.0,0.0,0.057727,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-03-01T09:15:15.110566+00:00,"Action Adventure, Pokémon FireRed and LeafGree...",Pokémon FireRed & LeafGreen inclui conteúdo au...,https://www.eurogamer.pt/pokemon-firered-leafg...,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-02-09T13:33:14.368225+00:00,"Third person, Racing, Nintendo, Nintendo Switc...",Switch 2 já alcançou 58% das vendas da PS5 em ...,https://www.eurogamer.pt/switch-2-ja-alcancou-...,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-03-10T14:42:15.059919+00:00,"Multiplayer Cooperative, Indie, PC, Slay the S...",Produtora de Slay the Spire 2 detesta micro-tr...,https://www.eurogamer.pt/produtora-de-slay-the...,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,data,tags,titulo,url,bow_aaa,bow_aang,bow_aaron,bow_abaixo,bow_abandona,bow_abandonado,...,bow_zonas,bow_zone,bow_zootopia,bow_zootropolis,bow_zor,bow_zoro,bow_zur,bow_zwei,bow_zygarde,palavras_unicas
0,2026-02-03T16:35:31.309576+00:00,"Android, iOS",Tecmo Bowl será adaptado para filme,https://www.eurogamer.pt/tecmo-bowl-sera-adapt...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,60
1,2026-04-02T07:40:11.552201+00:00,"Action Adventure, Platformer, Nintendo Switch,...",Fãs adoram Super Mario Galaxy: O Filme,https://www.eurogamer.pt/fas-adoram-super-mari...,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,79
2,2026-03-01T09:15:15.110566+00:00,"Action Adventure, Pokémon FireRed and LeafGree...",Pokémon FireRed & LeafGreen inclui conteúdo au...,https://www.eurogamer.pt/pokemon-firered-leafg...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,59
3,2026-02-09T13:33:14.368225+00:00,"Third person, Racing, Nintendo, Nintendo Switc...",Switch 2 já alcançou 58% das vendas da PS5 em ...,https://www.eurogamer.pt/switch-2-ja-alcancou-...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,89
4,2026-03-10T14:42:15.059919+00:00,"Multiplayer Cooperative, Indie, PC, Slay the S...",Produtora de Slay the Spire 2 detesta micro-tr...,https://www.eurogamer.pt/produtora-de-slay-the...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,87


In [38]:

from sklearn.feature_extraction.text import TfidfTransformer

transformer = TfidfTransformer()

matriz_tfidf = transformer.fit_transform(df_bow)

df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    columns = df_bow.columns)

print(df_tfidf.head())

   aaa  aang  aaron    abaixo  abandona  abandonado  abandonar  abandono  \
0  0.0   0.0    0.0  0.000000       0.0         0.0        0.0       0.0   
1  0.0   0.0    0.0  0.127451       0.0         0.0        0.0       0.0   
2  0.0   0.0    0.0  0.000000       0.0         0.0        0.0       0.0   
3  0.0   0.0    0.0  0.000000       0.0         0.0        0.0       0.0   
4  0.0   0.0    0.0  0.000000       0.0         0.0        0.0       0.0   

   abandonou  abby  ...  zona  zonas  zone  zootopia  zootropolis  zor  zoro  \
0        0.0   0.0  ...   0.0    0.0   0.0       0.0          0.0  0.0   0.0   
1        0.0   0.0  ...   0.0    0.0   0.0       0.0          0.0  0.0   0.0   
2        0.0   0.0  ...   0.0    0.0   0.0       0.0          0.0  0.0   0.0   
3        0.0   0.0  ...   0.0    0.0   0.0       0.0          0.0  0.0   0.0   
4        0.0   0.0  ...   0.0    0.0   0.0       0.0          0.0  0.0   0.0   

   zur  zwei  zygarde  
0  0.0   0.0      0.0  
1  0.0   0.0  

In [39]:
df_tfidf.sum(axis=0).sort_values()

stalker       0.058887
cost          0.058887
deathwish     0.078516
prologue      0.079205
cain          0.079205
               ...    
jogos        28.026977
the          29.226389
pokemon      31.659617
nintendo     31.885363
jogo         44.176963
Length: 9180, dtype: float64

In [40]:
# palavra com maior TF-IDF por notícia
maiores_palavras = df_tfidf.idxmax(axis=1)

print(maiores_palavras)

0            tecmo
1            filme
2            itens
3         nintendo
4             slay
           ...    
1244      petroleo
1245        mikami
1246    desempenho
1247      nintendo
1248        mahler
Length: 1249, dtype: object


In [41]:
maior_valor = df_tfidf.max(axis=1)

resultado = pd.DataFrame({
    "palavra": maiores_palavras,
    "tfidf": maior_valor
})

print(resultado.head())

    palavra     tfidf
0     tecmo  0.501552
1     filme  0.399734
2     itens  0.382076
3  nintendo  0.374182
4      slay  0.418965


In [42]:
from pathlib import Path

pasta_dados = Path("../dados")
pasta_dados.mkdir(exist_ok=True)

# Dados completos: metadados + texto original + texto limpo + texto sem stopwords
df_completo = df[["url", "titulo", "descricao", "tags", "data", "texto", "texto_limpo", "texto_sem_stopwords"]]
df_completo.to_csv(pasta_dados / "noticias.csv", index=False)

# Bag of Words (metadados + colunas bow_)
df_final.to_csv(pasta_dados / "bow.csv", index=False)

# TF-IDF (metadados + colunas tfidf_)
df_final_tfidf.to_csv(pasta_dados / "tfidf.csv", index=False)

print(f"Noticias completas: {pasta_dados / 'noticias.csv'} (shape={df_completo.shape})")
print(f"Bag of Words:       {pasta_dados / 'bow.csv'}      (shape={df_final.shape})")
print(f"TF-IDF:             {pasta_dados / 'tfidf.csv'}    (shape={df_final_tfidf.shape})")

Noticias completas: ../dados/noticias.csv (shape=(1249, 8))
Bag of Words:       ../dados/bow.csv      (shape=(1249, 9185))
TF-IDF:             ../dados/tfidf.csv    (shape=(1249, 9184))
